# step 4 — RQ3 인과 (지침 지시어 KV 치환, Spotlight 최종 판정)

**대응 RQ:** RQ3 **인과**. (directive-token = RQ3 관측 → step4 = RQ3 인과.)

**무엇을 확인하나.** stepC/step1은 **코드** 신호의 레버가 Key(어텐션)가 아니라 **Value**(내용)임을
인과로 보였다. step4는 **같은 KV 치환을 지침의 표기 지시어 토큰**(`camelCase`/`snake_case`)에
적용해, 지침의 레버가 Key인지 Value인지 (또는 아예 불활성인지)를 가른다. = 어텐션을 키우는
**Spotlight의 정면 인과 시험.**

**무엇을 바꾸나.** 지침 rule 문장의 지시어 토큰의 층 L KV를 **반대 지침**(선행은 그대로, 지침만
반대 표기로 렌더한 별도 forward)의 값으로 치환한다. 텍스트(`camelCase`)는 그대로, 모델 내부가
읽는 지침만 반대로. 경로 3종(Key/Value/Key+Value) × 전 층 스윕.

**무엇을 재나.** 세 상태 준수 선호 점수 `S = logP(target 이름) − logP(위반 이름)`:
`S_base`(조건 지침 그대로) · `S_clean`(반대 지침 그대로=천장) · `S_int`(지시어 KV만 반대로 치환).
**전이율 = (S_int − S_base)/(S_clean − S_base)** = 지침 KV를 반대로 바꿨을 때 행동이 반대 지침
쪽으로 얼마나 넘어가나. (층 × kind)마다 산출.

**조건 축 — 선행 2칸 대조.** 지침 레버가 잘 듣는 칸(균형 6/6)과 눌리는 칸(전부 위반)을 함께 태워,
"그 조건에서만" 반박을 미리 닫는다. 방향 2종(camel→snake, snake→camel)으로 대칭 확인.

**결과 분기(Spotlight 최종 판정).**
- **Value** 치환에 전이 O → 지침 레버=내용(Value) → Spotlight(Key 증폭) 구조적 실패, 방법론은 Value.
- **Key** 치환에 전이 O → 지침 레버=어텐션(Key) → Spotlight 유효 여지.
- **둘 다 무반응** → 지침 토큰 인과적 **불활성**(읽히는데 죽은 신호 = 가장 강한 반박).

설계·예측: `docs/step4/plan.md`.

> **메모리(T4):** output_attentions 안 씀(KV 캐시 편집) → eager 불필요, 가볍다. 스윕은 조건당
> 선행/반대지침 프롬프트를 각 1회 forward하고 work 캐시를 층·kind마다 편집→측정→복구로 재사용.
> **재개 가능:** 조건마다 저장, 이미 저장된 조건은 건너뜀. GPU 없으면 매우 느림.
> **정렬 주의:** `camelCase`/`snake_case`는 토큰 수가 달라 `token_unit='last'`(항상 1:1)를 기본으로
> 쓴다. 참고로 `'all'`(전체 서브토큰, 수 다르면 스킵)도 함께 돌려 `n_substituted`로 대조한다.
> **sanity:** 셀 6에서 균형 6/6 칸은 `S_base`와 `S_clean`이 방향대로 벌어지는지(레버 틈) 확인.


In [ ]:
# 환경 설정 — 설치, GPU 확인, 시드 고정
!pip install -q transformers accelerate torch matplotlib pandas

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)


In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin step4/instruction-kv
!git checkout step4/instruction-kv
!git pull --quiet origin step4/instruction-kv
!pip install -e . -q
import sys; sys.path.insert(0, 'src')


In [ ]:
# 조건 설정 — 지침 지시어 KV 치환 스윕. 방향 2 x 선행 2 x token_unit 2 x seed.
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation,
                                Intervention, InterventionKind)

MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')

# 방향: 조건 지침의 목표 표기. camel이면 camel->snake 치환(반대=snake), snake면 반대.
DIRECTIONS = [Notation.CAMEL, Notation.SNAKE]
# 선행 2칸: 균형 6/6(지침 레버 틈 큼) vs 전부 위반 0(지침이 눌림). 대조.
PRECEDING_NC = [6, 0]
# 정렬 단위: 'last'(항상 1:1, 기본) + 'all'(전체 서브토큰, 수 다르면 스킵 -> n_substituted로 대조).
TOKEN_UNITS = ['last', 'all']
SEEDS = list(range(5))

def sweep_cond(target, nc, tok_unit, s):
    return Condition(
        model=MODEL,
        preceding=PrecedingCode(n_compliant=nc, n_functions=12, composition=Composition.POOL),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=target),
        intervention=Intervention(kind=InterventionKind.KEY_VALUE, layers='sweep',
                                  target='instruction'),   # <- 지침 지시어 타깃(step4)
        seed=s, token_unit=tok_unit,
    )

sweep_conditions = [sweep_cond(t, nc, tu, s)
                    for t in DIRECTIONS for nc in PRECEDING_NC
                    for tu in TOKEN_UNITS for s in SEEDS]
print('조건 수:', len(sweep_conditions),
      '=', len(DIRECTIONS), 'x', len(PRECEDING_NC), 'x', len(TOKEN_UNITS), 'x', len(SEEDS))

PREDICTION = ('directive-token 관측: 지시어는 읽힌다. 인과 예측(불확실): 코드가 Value였으니 지침도 '
              'Value 치환에서 전이가 더 크고, Key 치환은 약할 것. 단 지침 토큰이 불활성이면 둘 다 '
              '무반응(가장 강한 Spotlight 반박). 균형 6/6에서 전이 O, 전부위반에서 약함이면 지침은 '
              '약한 레버. 예상과 달라도 조건 바꿔 맞추지 않고 그대로 기록(CLAUDE.md §4).')
print(sweep_conditions[0].slug())


In [ ]:
# 실행 — 조건별 즉시 저장(재개). 지침 지시어 KV 스윕(전 층 x K/V).
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model

handle = load_model(MODEL)   # output_attentions 안 씀 -> eager 불필요
print('layers:', handle.num_layers,
      '| L25 상대 위치:', round(handle.relative_layer(25), 3),
      '| GQA:', handle.gqa_info())

new = skipped = 0
for i, c in enumerate(sweep_conditions, 1):
    if result_path(c, step='step4').exists():
        skipped += 1
    else:
        out = run(c, handle=handle)   # intervention.kind!=NONE + sweep -> 개입 스윕(지침 타깃)
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step='step4', rq='RQ3', prediction=PREDICTION))
        new += 1
    if i % 5 == 0 or i == len(sweep_conditions):
        print(f'[{i}/{len(sweep_conditions)}] 새 {new} / 건너뜀 {skipped}')
print('완료.')


In [ ]:
# 결과 로드
from harness import result_path
from harness.results import load_result
recs = [load_result(result_path(c, step='step4')) for c in sweep_conditions]
print('로드:', len(recs), '-> results/step4/')
# 정렬 스킵 점검: token_unit='all'에서 지시어 토큰 수가 달라 스킵되면 n_substituted=0.
for r in recs[:4]:
    e = r.metrics.extra
    print(r.condition.token_unit, '| n_substituted:', e.get('n_substituted_tokens'),
          '| skipped:', e.get('skipped_names'), '|', e.get('viol_names'), '->', e.get('donor_names'))


In [ ]:
# 요약 — (층 x kind) 전이율 곡선을 (방향 x 선행)별로. 피크 표 + S 세 상태(레버 틈).
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import defaultdict
from harness.intervention import peak_layer

KINDS = ['key', 'value', 'key_value']
TU = 'last'   # 기본 정렬 단위(항상 1:1). 'all'은 아래 표에서 n_substituted로 대조.

def dir_label(t): return 'camel->snake' if t == 'camel' else 'snake->camel'
def pre_label(nc): return 'balanced 6/6' if nc == 6 else 'all-violation 0/12'

# (방향, 선행) -> kind -> {layer: [seed별 전이율]}
cells_agg = defaultdict(lambda: {k: defaultdict(list) for k in KINDS})
S_states = defaultdict(lambda: {'S_base': [], 'S_clean': []})
for r in recs:
    if r.condition.token_unit != TU:
        continue
    key = (r.condition.instruction.target_notation.value, r.condition.preceding.n_compliant)
    S_states[key]['S_base'].append(r.metrics.extra['S_base'])
    S_states[key]['S_clean'].append(r.metrics.extra['S_clean'])
    for L, flat in r.metrics.per_layer.items():
        for k in KINDS:
            kk = f'{k}__recovery'
            if kk in flat:
                cells_agg[key][k][int(L)].append(flat[kk])

def curve(key, k):
    layers = sorted(cells_agg[key][k])
    return layers, [float(np.mean(cells_agg[key][k][L])) for L in layers]

# S 세 상태(레버 틈) + 피크 전이율 표
print('=== 레버 틈(S_clean - S_base)과 피크 전이율 (token_unit=%s) ===' % TU)
rows = []
for key in sorted(S_states):
    t, nc = key
    sb = float(np.mean(S_states[key]['S_base'])); sc = float(np.mean(S_states[key]['S_clean']))
    for k in KINDS:
        layers, vals = curve(key, k)
        pk = peak_layer(dict(zip(layers, vals))) if vals else None
        rows.append({'direction': dir_label(t), 'preceding': pre_label(nc), 'kind': k,
                     'S_base': round(sb, 2), 'S_clean': round(sc, 2), 'gap': round(sc - sb, 2),
                     'peak_layer': pk[0] if pk else None,
                     'peak_transition': round(pk[1], 3) if pk else None})
df = pd.DataFrame(rows)
print(df.to_string(index=False))

# 'all' 정렬 단위 대조(스킵 여부)
alln = [r.metrics.extra.get('n_substituted_tokens') for r in recs if r.condition.token_unit == 'all']
print('\ntoken_unit=all n_substituted 분포:', sorted(set(alln)),
      '(0이면 지시어 토큰 수 불일치로 전체 스킵 -> last 결과를 주로 해석)')

# 플롯: (방향 x 선행) 4칸, 각 칸에 kind 3곡선(전이율 vs 층)
keys = sorted(cells_agg)
colors = {'key': '#2563C9', 'value': '#C6552B', 'key_value': '#3B8A45'}
fig, axes = plt.subplots(1, len(keys), figsize=(4.2*len(keys), 3.8), sharey=True)
if len(keys) == 1: axes = [axes]
for ax, key in zip(axes, keys):
    t, nc = key
    for k in KINDS:
        layers, vals = curve(key, k)
        ax.plot(layers, vals, color=colors[k], label=k, linewidth=1.8)
    ax.axhline(0, color='#999', linewidth=0.8, linestyle=':')
    ax.axhline(1, color='#999', linewidth=0.8, linestyle=':')
    ax.set_title('%s | %s' % (dir_label(t), pre_label(nc)), fontsize=10)
    ax.set_xlabel('layer')
axes[0].set_ylabel('transition rate')
axes[0].legend(fontsize=8, loc='best')
fig.suptitle('step4: instruction directive KV substitution -> behavior transition (token_unit=%s)' % TU,
             fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig('step4_transition_curves.png', dpi=130, bbox_inches='tight')
plt.show()
print('저장: step4_transition_curves.png')


In [ ]:
# 결과 다운로드
import shutil
shutil.make_archive('step4_results', 'zip', 'results/step4')
try:
    from google.colab import files
    files.download('step4_results.zip')
except Exception as e:
    print('Colab 아님(수동 다운로드): step4_results.zip', e)
